In [1]:
import os
import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
df = pl.read_csv("../data/diabetic_data.csv")
df.head()

encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
i64,i64,str,str,str,str,i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
2278392,8222157,"""Caucasian""","""Female""","""[0-10)""","""?""",6,25,1,1,"""?""","""Pediatrics-Endocrinology""",41,0,1,0,0,0,"""250.83""","""?""","""?""",1,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""NO"""
149190,55629189,"""Caucasian""","""Female""","""[10-20)""","""?""",1,1,7,3,"""?""","""?""",59,0,18,0,0,0,"""276""","""250.01""","""255""",9,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Up""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""",""">30"""
64410,86047875,"""AfricanAmerican""","""Female""","""[20-30)""","""?""",1,1,7,2,"""?""","""?""",11,5,13,2,0,1,"""648""","""250""","""V27""",6,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""NO"""
500364,82442376,"""Caucasian""","""Male""","""[30-40)""","""?""",1,1,7,2,"""?""","""?""",44,1,16,0,0,0,"""8""","""250.43""","""403""",7,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Up""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""","""NO"""
16680,42519267,"""Caucasian""","""Male""","""[40-50)""","""?""",1,1,7,1,"""?""","""?""",51,0,8,0,0,0,"""197""","""157""","""250""",5,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""","""NO"""


In [3]:
df.describe()

statistic,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
str,f64,f64,str,str,str,str,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""count""",101766.0,101766.0,"""101766""","""101766""","""101766""","""101766""",101766.0,101766.0,101766.0,101766.0,"""101766""","""101766""",101766.0,101766.0,101766.0,101766.0,101766.0,101766.0,"""101766""","""101766""","""101766""",101766.0,"""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766"""
"""null_count""",0.0,0.0,"""0""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0"""
"""mean""",1.6520e8,5.4330e7,null,null,null,null,2.024006,3.715642,5.754437,4.395987,null,null,43.095641,1.33973,16.021844,0.369357,0.197836,0.635566,null,null,null,7.422607,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",1.0264e8,3.8696e7,null,null,null,null,1.445403,5.280166,4.064081,2.985108,null,null,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,null,null,null,1.9336,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""",12522.0,135.0,"""?""","""Female""","""[0-10)""",""">200""",1.0,1.0,1.0,1.0,"""?""","""?""",1.0,0.0,1.0,0.0,0.0,0.0,"""10""","""11""","""11""",1.0,""">200""",""">7""","""Down""","""Down""","""Down""","""Down""","""Down""","""No""","""Down""","""Down""","""No""","""Down""","""Down""","""Down""","""Down""","""No""","""No""","""No""","""No""","""Down""","""Down""","""No""","""No""","""No""","""No""","""Ch""","""No""","""<30"""
"""25%""",8.4960072e7,2.3413212e7,null,null,null,null,1.0,1.0,1.0,2.0,null,null,31.0,0.0,10.0,0.0,0.0,0.0,null,null,null,6.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",1.5238968e8,4.5505143e7,null,null,null,null,1.0,1.0,7.0,4.0,null,null,44.0,1.0,15.0,0.0,0.0,0.0,null,null,null,8.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",2.30271972e8,8.7546186e7,null,null,null,null,3.0,4.0,7.0,6.0,null,null,57.0,2.0,20.0,0.0,0.0,1.0,null,null,null,9.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""",4.43867222e8,1.89502619e8,"""Other""","""Unknown/Invalid""","""[90-100)""","""[75-100)""",8.0,28.0,25.0,14.0,"""WC""","""Urology""",132.0,6.0,81.0,42.0,76.0,21.0,"""V71""","""V86""","""V86""",16.0,"""Norm""","""Norm""","""Up""","""Up""","""Up""","""Up""","""Up""","""Steady"""

In [4]:
X = df.drop(['readmitted'])
y = df['readmitted']

ordinal_cols = ['max_glu_serum', 'A1Cresult']

max_glu_categories    = [['None', 'Norm', '>200', '>300']]
A1Cresult_categories  = [['None', 'Norm', '>7', '>8']]

all_ordinal_categories = max_glu_categories + A1Cresult_categories


# Polars equivalent of select_dtypes
numerical_cols  = [col for col, dtype in zip(X.columns, X.dtypes) 
                   if dtype in [pl.Int64, pl.Float64, pl.Int32, pl.Float32]]
categorical_cols = [col for col, dtype in zip(X.columns, X.dtypes) 
                    if dtype in [pl.Utf8, pl.String, pl.Categorical, pl.Boolean]]
categorical_cols = [c for c in categorical_cols if c not in ordinal_cols]


In [5]:
# Split Test and Training data
# Convert polars DataFrame and Series to pandas DataFrame and Series
X_train, X_test, y_train, y_test = train_test_split(X.to_pandas(),
                                                    y.to_pandas(),
                                                    test_size=0.3,
                                                    random_state=42,
                                                    stratify=y)

# Split X_train further into 70% dev and 30% validation
X_dev, X_val, y_dev, y_val = train_test_split(X_train,
                                               y_train,
                                               test_size=0.3,
                                               random_state=42,
                                               stratify=y_train)

In [6]:
pl.DataFrame(X_train).describe()

statistic,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed
str,f64,f64,str,str,str,str,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""count""",71236.0,71236.0,"""71236""","""71236""","""71236""","""71236""",71236.0,71236.0,71236.0,71236.0,"""71236""","""71236""",71236.0,71236.0,71236.0,71236.0,71236.0,71236.0,"""71236""","""71236""","""71236""",71236.0,"""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236"""
"""null_count""",0.0,0.0,"""0""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0"""
"""mean""",1.6518e8,5.4216e7,null,null,null,null,2.020116,3.72125,5.7435,4.39466,null,null,43.112612,1.342523,16.016452,0.371104,0.1958,0.635353,null,null,null,7.418103,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",1.0271e8,3.8669e7,null,null,null,null,1.441682,5.291709,4.050373,2.979942,null,null,19.658896,1.705587,8.130801,1.269937,0.864571,1.269287,null,null,null,1.93048,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""",15738.0,135.0,"""?""","""Female""","""[0-10)""",""">200""",1.0,1.0,1.0,1.0,"""?""","""?""",1.0,0.0,1.0,0.0,0.0,0.0,"""10""","""11""","""11""",1.0,""">200""",""">7""","""Down""","""Down""","""Down""","""No""","""Down""","""No""","""Down""","""Down""","""No""","""Down""","""Down""","""Down""","""Down""","""No""","""No""","""No""","""No""","""Down""","""Down""","""No""","""No""","""No""","""No""","""Ch""","""No"""
"""25%""",8.4711426e7,2.3402142e7,null,null,null,null,1.0,1.0,1.0,2.0,null,null,31.0,0.0,10.0,0.0,0.0,0.0,null,null,null,6.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",1.52323008e8,4.5204345e7,null,null,null,null,1.0,1.0,7.0,4.0,null,null,44.0,1.0,15.0,0.0,0.0,0.0,null,null,null,8.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",2.30794458e8,8.7419277e7,null,null,null,null,3.0,4.0,7.0,6.0,null,null,57.0,2.0,20.0,0.0,0.0,1.0,null,null,null,9.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""",4.43867222e8,1.89502619e8,"""Other""","""Unknown/Invalid""","""[90-100)""","""[75-100)""",8.0,28.0,25.0,14.0,"""WC""","""Urology""",132.0,6.0,81.0,42.0,63.0,19.0,"""V71""","""V86""","""V86""",16.0,"""Norm""","""Norm""","""Up""","""Up""","""Up""","""Up""","""Up""","""Steady""","""Up""","""Up""","""Steady""","""Up""","""Up""","""Up""","""Up""","""No""","""Up""","""No""","""No""","""Up""","""Up""","""

In [7]:
# Build preprocessing pipeline
preprocessor = ColumnTransformer(transformers=[
    ('num', RobustScaler(), numerical_cols),
    ('ord', OrdinalEncoder(categories=all_ordinal_categories), ordinal_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

# Fit on train (dev) set only
X_dev_scaled  = preprocessor.fit_transform(X_dev)

# Transform all 3 dev, validation, and test sets
X_val_scaled  = preprocessor.transform(X_val)
X_test_scaled  = preprocessor.transform(X_test)

In [8]:
os.makedirs('../data', exist_ok=True)

# Use unscaled pandas DataFrames directly
dev_df  = pl.from_pandas(X_dev)
val_df  = pl.from_pandas(X_val)
test_df = pl.from_pandas(X_test)

# Add target column
dev_df  = dev_df.with_columns(pl.Series('readmitted', y_dev.to_list()))
val_df  = val_df.with_columns(pl.Series('readmitted', y_val.to_list()))
test_df = test_df.with_columns(pl.Series('readmitted', y_test.to_list()))

# Combine dev + val into full train
train_df = pl.concat([dev_df, val_df])

# Save all four
train_df.write_parquet('../data/train.parquet')
dev_df.write_parquet('../data/dev.parquet')
val_df.write_parquet('../data/val.parquet')
test_df.write_parquet('../data/test.parquet')

print(f"train.parquet : {train_df.shape}")
print(f"dev.parquet   : {dev_df.shape}")
print(f"val.parquet   : {val_df.shape}")
print(f"test.parquet  : {test_df.shape}")
print(f"\nSaved to: {os.path.abspath('../data/')}")

train.parquet : (71236, 50)
dev.parquet   : (49865, 50)
val.parquet   : (21371, 50)
test.parquet  : (30530, 50)

Saved to: /Users/eunsung/GitHub/dataquest-2026/data


In [9]:
files = ['train', 'dev', 'val', 'test']

for f in files:
    df = pl.read_parquet(f'../data/{f}.parquet')
    print(f'=== {f}.parquet {df.shape} ===')
    print(df.describe())
    print()

=== train.parquet (71236, 50) ===
shape: (9, 51)
┌────────────┬────────────┬────────────┬───────┬───┬────────────┬────────┬────────────┬────────────┐
│ statistic  ┆ encounter_ ┆ patient_nb ┆ race  ┆ … ┆ metformin- ┆ change ┆ diabetesMe ┆ readmitted │
│ ---        ┆ id         ┆ r          ┆ ---   ┆   ┆ pioglitazo ┆ ---    ┆ d          ┆ ---        │
│ str        ┆ ---        ┆ ---        ┆ str   ┆   ┆ ne         ┆ str    ┆ ---        ┆ str        │
│            ┆ f64        ┆ f64        ┆       ┆   ┆ ---        ┆        ┆ str        ┆            │
│            ┆            ┆            ┆       ┆   ┆ str        ┆        ┆            ┆            │
╞════════════╪════════════╪════════════╪═══════╪═══╪════════════╪════════╪════════════╪════════════╡
│ count      ┆ 71236.0    ┆ 71236.0    ┆ 71236 ┆ … ┆ 71236      ┆ 71236  ┆ 71236      ┆ 71236      │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0     ┆ … ┆ 0          ┆ 0      ┆ 0          ┆ 0          │
│ mean       ┆ 1.6518e8   ┆ 5.4216e7   ┆ n